# NCBI Genome Analysis Pipeline: Automated BLAST → FASTA Download → Augustus (with Large Sequence Splitting & Parallel Processing)

In [3]:
# NCBI Genome pipeline (Edge) — WITH LARGE SEQUENCE SPLITTING AND FOLDER STRUCTURE

from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time, os, glob, shutil
import pandas as pd
import traceback
import re
import math
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed


# ---------- Config ----------
EDGEDRIVER_PATH = r"C:\edgedriver_win64\msedgedriver.exe"
GENOME_HOME = "https://www.ncbi.nlm.nih.gov/datasets/genome/"
TIMEOUT = 30
BASE_DIR = r"D:\DOWNLOAD_SECTION\Augustus_csv"
AUGUSTUS_MAX_MB = 7  # Safe limit: 7 MB (8 MB se safe rahe)
CHUNK_SIZE_BP = 7000000  # ~7 MB (assuming 1 char = 1 byte)
AUGUSTUS_COMPLETE_TIMEOUT = 45  # 45 minutes wait for Augustus completion

# ================================================================
# FOLDER STRUCTURE FUNCTIONS
# ================================================================

def create_species_folders(species_name):
    """
    Create folder structure for a species:
    BASE_DIR/
    └── species_name/
        ├── excel/
        ├── fasta/
        ├── protein/
        └── nucleotide/
    """
    # Clean species name for folder
    safe_species = re.sub(r'[^\w\-_]', '_', species_name)
    
    # Create main species folder
    species_dir = os.path.join(BASE_DIR, safe_species)
    os.makedirs(species_dir, exist_ok=True)
    
    # Create subfolders
    excel_dir = os.path.join(species_dir, "excel")
    fasta_dir = os.path.join(species_dir, "fasta")
    protein_dir = os.path.join(species_dir, "protein")
    nucleotide_dir = os.path.join(species_dir, "nucleotide")
    
    os.makedirs(excel_dir, exist_ok=True)
    os.makedirs(fasta_dir, exist_ok=True)
    os.makedirs(protein_dir, exist_ok=True)
    os.makedirs(nucleotide_dir, exist_ok=True)
    
    print(f"\n📁 Folder structure created for {safe_species}:")
    print(f"   📊 Excel files: {excel_dir}")
    print(f"   🧬 FASTA files: {fasta_dir}")
    print(f"   🧪 Protein results: {protein_dir}")
    print(f"   🧪 Nucleotide results: {nucleotide_dir}")
    
    return {
        "species": safe_species,
        "base": species_dir,
        "excel": excel_dir,
        "fasta": fasta_dir,
        "protein": protein_dir,
        "nucleotide": nucleotide_dir
    }

def get_species_folder(species_name):
    """Get or create species folder structure"""
    return create_species_folders(species_name)

# ================================================================
# EDGE OPTIONS - FIXED: Main driver WITHOUT page_load_strategy
# ================================================================

def get_edge_options():
    """Main driver options - NO page_load_strategy to avoid extra tab"""
    edge_opts = Options()
    # ⭐ IMPORTANT FIX: REMOVED page_load_strategy - yeh extra tab create karta hai
    # edge_opts.page_load_strategy = "none"  # <-- REMOVED
    
    edge_opts.add_argument("--start-maximized")
    edge_opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    edge_opts.add_experimental_option("detach", True)
    prefs = {
        "download.default_directory": BASE_DIR,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    edge_opts.add_experimental_option("prefs", prefs)
    return edge_opts


# ================================================================
# NEW FUNCTION: Augustus driver options - WITH page_load_strategy
# ================================================================

def get_augustus_driver_options():
    """
    Augustus driver options - WITH page_load_strategy = "none"
    Sirf Augustus ke liye use karo to avoid blocking during long processing
    """
    aug_opts = Options()
    # ⭐ ONLY FOR AUGUSTUS: page_load_strategy = "none" to avoid blocking
    aug_opts.page_load_strategy = "none"
    
    aug_opts.add_argument("--start-maximized")
    aug_opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    aug_opts.add_experimental_option("detach", True)
    prefs = {
        "download.default_directory": BASE_DIR,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    aug_opts.add_experimental_option("prefs", prefs)
    return aug_opts


# ================================================================
# GENERIC HELPERS
# ================================================================

def wait_ready(driver, timeout=TIMEOUT):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

def smooth_scroll_page(driver, pause=0.8, max_loops=40):
    last_h = -1
    loops = 0
    while loops < max_loops:
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        last_h = h
        loops += 1

def scroll_virtual_grid(driver, pause=0.6, max_stall=4):
    try:
        scroller = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//div[contains(@class,'MuiDataGrid-virtualScroller')]"))
        )
    except Exception:
        return False
    stall = 0
    last_top = -1
    while stall < max_stall:
        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", scroller)
        time.sleep(pause)
        top = driver.execute_script("return arguments[0].scrollTop;", scroller)
        if top == last_top:
            stall += 1
        else:
            stall = 0
        last_top = top
    return True

def load_all_rows(driver):
    print("📜 Loading all rows…")
    used_grid = scroll_virtual_grid(driver)
    if not used_grid:
        smooth_scroll_page(driver)
    print("✅ Scrolling done")

def get_rows(driver):
    rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    if rows:
        return rows, "table"
    grid_rows = driver.find_elements(By.XPATH, "//div[@role='row' and .//a]")
    if grid_rows:
        return grid_rows, "grid"
    return [], "none"

def get_assembly_anchor(row):
    candidates = row.find_elements(
        By.XPATH,
        ".//a[contains(@href,'/genome/') or contains(@href,'assembly') or contains(@href,'/datasets/genome/')]"
    )
    for a in candidates:
        txt = (a.text or "").strip()
        if txt:
            return a, txt
    anchors = row.find_elements(By.XPATH, ".//a")
    for a in anchors:
        txt = (a.text or "").strip()
        if txt:
            return a, txt
    return None, ""

def row_has_verified(row):
    try:
        row.find_element(By.CSS_SELECTOR, "svg[data-testid='VerifiedIcon']")
        return True
    except Exception:
        return False

def row_has_submitter(row):
    try:
        txt = (row.text or "").lower()
        return "submitter" in txt
    except Exception:
        return False

def highlight_and_click(driver, el):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    driver.execute_script("arguments[0].style.outline='3px solid #22c55e';", el)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", el)

def pick_and_click_by_priority(driver):
    rows, kind = get_rows(driver)
    if not rows:
        print("❌ No rows detected.")
        return False

    print(f"🧮 Rows detected: {len(rows)} ({kind})")

    bucket1, bucket2, bucket3 = [], [], []  # badge+submitter, badge, submitter

    for r in rows:
        has_badge = row_has_verified(r)
        has_subm  = row_has_submitter(r)
        a_el, a_name = get_assembly_anchor(r)
        if not a_el:
            continue
        item = (r, a_el, a_name)
        if has_badge and has_subm:
            bucket1.append(item)
        elif has_badge:
            bucket2.append(item)
        elif has_subm:
            bucket3.append(item)

    def click_first(bucket, label):
        if not bucket:
            return False
        _r, a_el, a_name = bucket[0]
        href = a_el.get_attribute("href")
        print(f"👉 {label}: Clicking assembly  →  {a_name}  ({href})")
        highlight_and_click(driver, a_el)
        return True

    if click_first(bucket1, "Priority 1 (Badge + Submitter)"): return True
    if click_first(bucket2, "Priority 2 (Badge only)"):        return True
    if click_first(bucket3, "Priority 3 (Submitter only)"):    return True

    print("ℹ️ No row matched any priority.")
    return False

def wait_url_change(driver, old_url, timeout=TIMEOUT):
    try:
        WebDriverWait(driver, timeout).until(lambda d: d.current_url != old_url)
        return True
    except Exception:
        return False

# ================================================================
# SPECIES SEARCH
# ================================================================

def select_exact_species(driver, species_name):
    print("🔎 Selecting species…")
    box = WebDriverWait(driver, TIMEOUT).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "input#taxonomy_autocomplete"))
    )
    box.clear()
    for ch in species_name:
        box.send_keys(ch)
        time.sleep(0.05)

    WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "ul[role='listbox'] li[role='option']"))
    )

    try:
        xpath = (
            "//li[contains(@class,'MuiAutocomplete-option') and "
            "translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz') = "
            f"translate('{species_name}', 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz')]"
        )
        exact = WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].click();", exact)
        print(f"✅ Exact match: {species_name}")
    except Exception:
        first_opt = driver.find_element(By.CSS_SELECTOR, "ul[role='listbox'] li[role='option']")
        driver.execute_script("arguments[0].click();", first_opt)
        print("⚠️ Exact match not found — first suggestion used")

def get_results_url(driver, species_name):
    select_exact_species(driver, species_name)
    btn = WebDriverWait(driver, TIMEOUT).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Search')]"))
    )
    driver.execute_script("arguments[0].click();", btn)
    WebDriverWait(driver, TIMEOUT).until(lambda d: "?taxon=" in d.current_url)
    print("🔗 Results URL:", driver.current_url)
    return driver.current_url

# ================================================================
# BLAST FUNCTIONS
# ================================================================

def click_blast_button(driver):
    print("🚀 Looking for BLAST button…")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(0.8)

    blast_xpath = (
        "//a[contains(translate(.,'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'blast the reference genome') "
        "or contains(translate(.,'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'blast the genome')]"
    )

    try:
        blast_btn = WebDriverWait(driver, TIMEOUT).until(
            EC.element_to_be_clickable((By.XPATH, blast_xpath))
        )
    except Exception:
        blast_btn = WebDriverWait(driver, TIMEOUT).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'BLAST')]")
        ))

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", blast_btn)
    driver.execute_script("arguments[0].style.outline='3px solid #f97316';", blast_btn)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", blast_btn)
    print("✅ BLAST button clicked.")

def switch_to_blast_tab(driver):
    for handle in driver.window_handles:
        driver.switch_to.window(handle)
        if "blast" in driver.current_url.lower() or "blast.cgi" in driver.current_url.lower():
            return True
    return False

def set_tblastn(driver):
    print("🎯 Setting tBLASTn…")
    switch_to_blast_tab(driver)
    wait_ready(driver)

    tried = []
    try:
        btn = WebDriverWait(driver, 6).until(
            EC.element_to_be_clickable((By.ID, "tblastnTab"))
        )
        driver.execute_script("arguments[0].click();", btn)
        tried.append("direct id tblastnTab")
    except:
        print("⚠️ tBLASTn button not found!")

    try:
        WebDriverWait(driver, 5).until(
            lambda d: "PROGRAM=tblastn" in d.current_url or "tblastn" in d.page_source.lower()
        )
        print(f"✅ tBLASTn set (methods tried: {', '.join(tried)}).")
    except:
        print(f"⚠️ tBLASTn may not be set (methods tried: {', '.join(tried)}).")

def paste_query_sequence(driver, fasta_seq):
    print("📝 Pasting query sequence…")
    switch_to_blast_tab(driver)
    wait_ready(driver)

    try:
        box = WebDriverWait(driver, TIMEOUT).until(
            EC.presence_of_element_located((By.XPATH, "//textarea[@id='seq' or @name='QUERY']"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", box)
        driver.execute_script("arguments[0].style.outline='3px solid #3b82f6';", box)
        box.clear()
        box.send_keys(fasta_seq)
        print("✅ Query pasted successfully.")
    except Exception as e:
        print(f"❌ Could not paste query: {e}")

def click_final_blast(driver):
    print("🔥 Clicking final BLAST button…")
    switch_to_blast_tab(driver)
    wait_ready(driver)

    try:
        blast_btn = WebDriverWait(driver, TIMEOUT).until(
            EC.presence_of_element_located(
                (By.XPATH, "//div[@id='blastButton1']//input[@type='button' and @value='BLAST']")
            )
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", blast_btn)
        driver.execute_script("arguments[0].style.outline='3px solid #22c55e';", blast_btn)
        time.sleep(0.4)
        driver.execute_script("arguments[0].click();", blast_btn)
        print("✅ Final BLAST started.")
    except Exception as e:
        print(f"❌ Could not click final BLAST: {e}")

def click_alignments_button(driver):
    print("📊 Waiting for BLAST results and Alignments button…")
    switch_to_blast_tab(driver)
    try:
        align_btn = WebDriverWait(driver, 300).until(   # BLAST can take a few mins
            EC.element_to_be_clickable((By.ID, "btnAlign"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", align_btn)
        driver.execute_script("arguments[0].style.outline='3px solid #e11d48';", align_btn)
        time.sleep(0.4)
        driver.execute_script("arguments[0].click();", align_btn)
        print("✅ Alignments tab clicked.")
    except Exception as e:
        print(f"❌ Could not click Alignments: {e}")

# ================================================================
# DOWNLOAD HELPERS
# ================================================================

def click_download_button(driver):
    print("💾 Clicking 'Download' button…")
    try:
        dl_btn = WebDriverWait(driver, TIMEOUT).until(
            EC.element_to_be_clickable((By.ID, "btnDwnldAln"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", dl_btn)
        driver.execute_script("arguments[0].style.outline='3px solid #a855f7';", dl_btn)
        time.sleep(0.4)
        driver.execute_script("arguments[0].click();", dl_btn)
        print("✅ Download button clicked.")
    except Exception as e:
        print(f"❌ Could not click Download button: {e}")

def download_hit_table(driver):
    print("📥 Downloading Hit Table (CSV)…")
    try:
        csv_link = WebDriverWait(driver, TIMEOUT).until(
            EC.element_to_be_clickable((By.ID, "dwHitCsvAln"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", csv_link)
        driver.execute_script("arguments[0].style.outline='3px solid #06b6d4';", csv_link)
        time.sleep(0.4)
        driver.execute_script("arguments[0].click();", csv_link)
        print(f"✅ CSV download started → {BASE_DIR}")
    except Exception as e:
        print(f"❌ Could not download CSV: {e}")

def rename_latest_csv(species_name, before_files, folders):
    """Sirf nayi download huwi CSV ko species ke naam se rename karega and move to excel folder"""
    try:
        new_file = None
        # wait upto 60 sec for new file
        for _ in range(60):
            after_files = set(glob.glob(os.path.join(BASE_DIR, "*.csv")))
            new_files = after_files - before_files
            if new_files:
                new_file = list(new_files)[0]
                break
            time.sleep(1)

        if not new_file:
            print("❌ No new CSV file detected.")
            return None

        safe_name = species_name.replace(" ", "_") + ".csv"
        new_path = os.path.join(folders["excel"], safe_name)
        
        # Remove if exists
        if os.path.exists(new_path):
            os.remove(new_path)
            
        shutil.move(new_file, new_path)
        print(f"✅ CSV renamed and moved to: {new_path}")
        return new_path
    except Exception as e:
        print(f"❌ Rename error: {e}")
        return None

# ================================================================
# ENHANCED FUNCTIONS FOR LARGE SEQUENCE HANDLING
# ================================================================

def wait_for_fasta_download(before_files, timeout=None):
    """
    UNLIMITED WAIT - FASTA file download complete hone tak wait karo
    timeout=None means wait forever
    """
    print(f"⏳ Waiting for download to complete (unlimited time)...")
    
    start_time = time.time()
    last_size = 0
    stall_count = 0
    
    while True:  # Infinite loop
        # Check for .fa, .fasta, .txt files
        after_files = set(glob.glob(os.path.join(BASE_DIR, "*.fa"))) | \
                      set(glob.glob(os.path.join(BASE_DIR, "*.fasta"))) | \
                      set(glob.glob(os.path.join(BASE_DIR, "*.txt")))
        new_files = after_files - before_files
        
        if new_files:
            for file in new_files:
                if not file.endswith(('.crdownload', '.tmp', '.part')):
                    # Check if file size is stable
                    current_size = os.path.getsize(file)
                    
                    if current_size == last_size:
                        stall_count += 1
                        if stall_count > 3:  # Stable for 3 checks
                            if current_size > 0:
                                elapsed = int(time.time() - start_time)
                                hours = elapsed // 3600
                                minutes = (elapsed % 3600) // 60
                                seconds = elapsed % 60
                                print(f"   ✅ Download complete after {hours:02d}:{minutes:02d}:{seconds:02d}")
                                print(f"   📊 File size: {current_size / (1024 * 1024):.2f} MB")
                                return file
                    else:
                        last_size = current_size
                        stall_count = 0
                        size_mb = current_size / (1024 * 1024)
                        print(f"   📥 Downloaded: {size_mb:.2f} MB")
        
        # Show elapsed time every minute
        if int(time.time() - start_time) % 60 == 0:
            elapsed = int(time.time() - start_time)
            hours = elapsed // 3600
            minutes = (elapsed % 3600) // 60
            seconds = elapsed % 60
            print(f"   ⏱️ Downloading... {hours:02d}:{minutes:02d}:{seconds:02d} elapsed")
        
        time.sleep(2)

def read_fasta_file(file_path):
    """
    Downloaded FASTA file ko read karo aur SIRF base pairs return karo (header hatao)
    Memory efficient reading for large files
    """
    print(f"\n📖 Reading FASTA file: {file_path}")
    
    try:
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        print(f"   File size: {file_size:.2f} MB")
        
        if file_size > 100:  # 100 MB se bada file
            print("   ⚠️ Large file detected, using memory-efficient reading")
            return read_large_fasta_file(file_path)
        
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Extract header and sequence
        header, base_pairs, bp_count = extract_sequence_only(content)
        
        print(f"✅ File read: {bp_count:,} base pairs")
        print(f"   Original header: {header[:100]}..." if len(header) > 100 else f"   Original header: {header}")
        
        # Return SIRF base pairs
        return base_pairs
        
    except Exception as e:
        print(f"❌ Error reading FASTA file: {e}")
        return None

def read_large_fasta_file(file_path):
    """
    Memory-efficient reading for very large FASTA files
    """
    try:
        header = ""
        base_pairs = []
        total_bp = 0
        
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                    
                if line.startswith('>') and line_num == 0:
                    header = line
                elif line.startswith('>'):
                    # New header - ignore for now (we only want first sequence)
                    continue
                else:
                    # Sequence line - filter valid bases
                    valid_bases = re.sub(r'[^ACGTNacgtn]', '', line)
                    base_pairs.append(valid_bases)
                    total_bp += len(valid_bases)
                    
                    # Progress indicator for very large files
                    if total_bp % 1000000 == 0:
                        print(f"   ...read {total_bp/1000000:.1f}M bp")
        
        sequence_only = ''.join(base_pairs)
        print(f"✅ Large file read: {total_bp:,} base pairs")
        print(f"   Header: {header[:100]}..." if len(header) > 100 else f"   Header: {header}")
        
        return sequence_only
        
    except Exception as e:
        print(f"❌ Error reading large FASTA file: {e}")
        return None

def download_full_fasta_sequence(driver, species_name, begin, end, folders, is_positive):
    """
    NCBI GenBank page se "Send to" -> "File" -> "Create File" click karke
    FASTA file download karo (jisme poori sequence hogi)
    MODIFIED: is_positive parameter added for file naming
    """
    print("\n" + "="*80)
    print("📥 DOWNLOADING FULL FASTA SEQUENCE VIA SEND TO")
    print("="*80)
    
    try:
        # Step 1: Click "Send to" button
        print("🔍 Looking for 'Send to' button...")
        send_to_btn = WebDriverWait(driver, TIMEOUT).until(
            EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'Send to:') or contains(@class, 'tgt_dark')]"))
        )
        highlight_and_click(driver, send_to_btn)
        print("✅ 'Send to' clicked")
        time.sleep(2)
        
        # Step 2: Select "File" radio button
        print("🔍 Selecting 'File' radio button...")
        file_radio = WebDriverWait(driver, TIMEOUT).until(
            EC.element_to_be_clickable((By.ID, "dest_File"))
        )
        driver.execute_script("arguments[0].click();", file_radio)
        print("✅ 'File' radio selected")
        time.sleep(1)
        
        # Step 3: Click "Create File" button
        print("🔍 Clicking 'Create File' button...")
        create_btn = WebDriverWait(driver, TIMEOUT).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'button_apply') and contains(text(), 'Create File')]"))
        )
        
        # Get current files before download
        before_files = set(glob.glob(os.path.join(BASE_DIR, "*.fa"))) | \
                       set(glob.glob(os.path.join(BASE_DIR, "*.fasta"))) | \
                       set(glob.glob(os.path.join(BASE_DIR, "*.txt")))
        
        highlight_and_click(driver, create_btn)
        print("✅ 'Create File' clicked - download started")
        
        # Step 4: Wait for download to complete
        print("⏳ Waiting for FASTA download to complete...")
        fasta_path = wait_for_fasta_download(before_files, timeout=None)
        
        if fasta_path:
            print(f"✅ FASTA downloaded: {fasta_path}")
            
            # Rename file with species, range, and positive/negative indicator
            safe_species = species_name.replace(" ", "_")
            case_prefix = "positive" if is_positive else "negative"
            new_name = f"{case_prefix}_{safe_species}_{begin}_{end}.fasta"
            new_path = os.path.join(folders["fasta"], new_name)
            
            # Remove if already exists
            if os.path.exists(new_path):
                os.remove(new_path)
                
            shutil.move(fasta_path, new_path)
            print(f"✅ FASTA moved to: {new_path}")
            
            return new_path
        else:
            print("❌ FASTA download timeout or failed")
            return None
            
    except Exception as e:
        print(f"❌ Error downloading FASTA: {e}")
        traceback.print_exc()
        return None

# ================================================================
# ENHANCED SEQUENCE ANALYSIS AND PASTING FUNCTIONS
# ================================================================

def extract_sequence_only(fasta_text):
    """
    FAST header ko ALAG karo aur sirf sequence (base pairs) count karo
    Returns: (header, sequence_only, base_pair_count)
    """
    if not fasta_text:
        return "", "", 0
    
    lines = fasta_text.strip().split('\n')
    
    # Pehli line header hai
    header = lines[0] if lines and lines[0].startswith('>') else ""
    
    # Baaki sab sequence hai (base pairs)
    sequence_lines = lines[1:] if header else lines
    
    # Sirf ACGTN characters rakho, whitespace hatao
    sequence_only = ''.join([line.strip() for line in sequence_lines if line.strip()])
    
    # Sirf valid base pairs count karo (A, C, G, T, N - case insensitive)
    base_pairs = re.sub(r'[^ACGTNacgtn]', '', sequence_only)
    base_pair_count = len(base_pairs)
    
    print(f"\n{'='*80}")
    print(f"🔬 SEQUENCE ANALYSIS")
    print(f"{'='*80}")
    print(f"   Header: {header[:100]}..." if len(header) > 100 else f"   Header: {header}")
    print(f"   Total lines: {len(lines)}")
    print(f"   Sequence lines: {len(sequence_lines)}")
    print(f"   Raw sequence length: {len(sequence_only):,} chars")
    print(f"   Valid base pairs (ACGTN): {base_pair_count:,}")
    print(f"{'='*80}\n")
    
    return header, base_pairs, base_pair_count

# ================================================================
# NEW FUNCTION: Split large sequence into Augustus-compatible chunks
# ================================================================

def split_sequence_for_augustus(fasta_sequence, species_name, begin, end):
    """
    Large sequence ko chhote chunks mein split karo jo Augustus accept karega
    Returns: list of (chunk_sequence, chunk_begin, chunk_end)
    """
    print("\n" + "="*80)
    print("🔪 SPLITTING LARGE SEQUENCE FOR AUGUSTUS")
    print("="*80)
    
    # Extract header and base pairs
    header, base_pairs, total_bp = extract_sequence_only(fasta_sequence)
    
    # Calculate size in MB
    size_mb = total_bp / (1024 * 1024)
    print(f"📊 Total sequence: {total_bp:,} bp ({size_mb:.2f} MB)")
    print(f"📊 Augustus limit: {AUGUSTUS_MAX_MB} MB")
    
    if size_mb <= AUGUSTUS_MAX_MB:
        print("✅ Sequence within limit - no splitting needed")
        return [(fasta_sequence, begin, end)]
    
    # Calculate number of chunks
    num_chunks = math.ceil(size_mb / AUGUSTUS_MAX_MB)
    chunk_size_bp = min(CHUNK_SIZE_BP, total_bp // num_chunks + 100000)
    
    print(f"📊 Splitting into {num_chunks} chunks")
    print(f"📊 Chunk size: ~{chunk_size_bp / (1024*1024):.2f} MB ({chunk_size_bp:,} bp)")
    
    chunks = []
    total_range = end - begin + 1
    
    for i in range(num_chunks):
        # Calculate chunk boundaries
        chunk_start_bp = i * chunk_size_bp
        chunk_end_bp = min((i + 1) * chunk_size_bp, total_bp)
        
        # Calculate genomic coordinates
        chunk_begin = begin + chunk_start_bp
        chunk_end = begin + chunk_end_bp - 1
        
        # Extract sequence for this chunk
        chunk_sequence = base_pairs[chunk_start_bp:chunk_end_bp]
        
        # Create header for this chunk
        chunk_header = f">{species_name}_{chunk_begin}_{chunk_end} (part {i+1}/{num_chunks})"
        chunk_fasta = chunk_header + '\n' + chunk_sequence
        
        chunk_size = len(chunk_sequence) / (1024*1024)
        print(f"   Chunk {i+1}: {chunk_begin:,}-{chunk_end:,} ({chunk_size:.2f} MB)")
        
        chunks.append((chunk_fasta, chunk_begin, chunk_end))
    
    return chunks

# ================================================================
# ⭐ FIXED FUNCTION 1: wait_for_augustus_done - UNTITLED5 VERSION
# ================================================================

def wait_for_augustus_done(driver, folders, chunk_num, max_wait_minutes=45):
    """
    UNTITLED5 VERSION - Sirf "Done." text check karta hai, no stuck detection
    """
    print("\n⏳ Augustus server processing sequence...")
    print(f"   Waiting for 'Done.' text (max {max_wait_minutes} minutes)")
    
    start_time = time.time()
    max_wait_seconds = max_wait_minutes * 60
    
    while time.time() - start_time < max_wait_seconds:
        elapsed = int(time.time() - start_time)
        minutes = elapsed // 60
        seconds = elapsed % 60
        
        try:
            page_source = driver.page_source.lower()
            
            # Check for "Done." text
            if re.search(r'<p>\s*done\.\s*</p>', page_source, re.IGNORECASE) or \
               ("done." in page_source and "submit another job" in page_source.lower()):
                print(f"✅ 'Done.' text detected after {minutes}m {seconds}s")
                return True
            
            # Progress update every minute
            if minutes > 0 and minutes % 1 == 0 and seconds < 10:
                print(f"   Still processing... {minutes}m {seconds}s elapsed")
            
            time.sleep(10)
            
        except Exception as e:
            print(f"   Check error: {e}")
            time.sleep(10)
    
    print(f"❌ Max wait time ({max_wait_minutes} minutes) exceeded - no 'Done.' detected")
    
    # Save debug info
    try:
        debug_file = os.path.join(folders["base"], f"timeout_chunk{chunk_num}_{int(time.time())}.html")
        with open(debug_file, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"   Debug page saved: {debug_file}")
    except:
        pass
    
    return False

# ================================================================
# ⭐ FIXED FUNCTION 2: check_augustus_results_ready - UNTITLED5 VERSION
# ================================================================

def check_augustus_results_ready(driver, folders, chunk_num, max_wait_seconds=120):
    """
    UNTITLED5 VERSION - Checks for both required links
    """
    print(f"\n⏳ Checking Augustus results page (max {max_wait_seconds//60} minute)...")
    
    start_time = time.time()
    check_interval = 10
    
    # Expected links
    amino_link_text = "predicted amino acid sequences"
    coding_link_text = "predicted coding sequences"
    
    while time.time() - start_time < max_wait_seconds:
        elapsed = int(time.time() - start_time)
        
        try:
            page_source = driver.page_source.lower()
            
            # Check if both links exist
            has_amino = amino_link_text.lower() in page_source
            has_coding = coding_link_text.lower() in page_source
            
            # Scenario 1: Dono links hain → SUCCESS
            if has_amino and has_coding:
                print(f"✅ Both links detected after {elapsed} seconds")
                print(f"   ✓ {amino_link_text}")
                print(f"   ✓ {coding_link_text}")
                return True
            
            # Progress update
            if elapsed % 30 == 0:
                print(f"   Still waiting... ({elapsed}s) - Found: Amino={has_amino}, Coding={has_coding}")
            
            time.sleep(check_interval)
            
        except Exception as e:
            print(f"   Error checking page: {e}")
            time.sleep(check_interval)
    
    # Timeout - Scenario 2: Sirf text results ya incomplete
    print(f"\n⚠️ TIMEOUT after {max_wait_seconds//60} minute - Required links not found")
    print(f"   Amino acids: {'✅' if amino_link_text.lower() in driver.page_source.lower() else '❌'}")
    print(f"   Coding seq: {'✅' if coding_link_text.lower() in driver.page_source.lower() else '❌'}")
    print(f"   Closing stuck tab and moving to next chunk...")
    
    # Save debug info
    try:
        debug_file = os.path.join(folders["base"], f"failed_chunk{chunk_num}_{int(time.time())}.html")
        with open(debug_file, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"   Debug page saved: {debug_file}")
    except:
        pass
    
    return False


# ================================================================
# FIXED: AUGUSTUS FUNCTIONS WITH BETTER ERROR HANDLING
# ================================================================

def paste_sequence_verified(driver, fasta_sequence):
    """
    FIXED VERSION: window.open() use karta hai instead of driver.get()
    """
    AUGUSTUS_URL = "https://bioinf.uni-greifswald.de/augustus/submission.php"
    
    if not fasta_sequence:
        print("❌ No sequence provided.")
        return False

    sequence_size_bytes = len(fasta_sequence.encode("utf-8"))
    sequence_size_mb = sequence_size_bytes / (1024 * 1024)
    print(f"📊 Sequence size: {sequence_size_mb:.2f} MB")

    # Web Augustus ke liye safe limit (7 MB - conservative)
    MAX_WEB_MB = 7
    if sequence_size_mb > MAX_WEB_MB:
        print(f"❌ Sequence too large: {sequence_size_mb:.2f} MB > {MAX_WEB_MB} MB")
        print("💡 Use split_sequence_for_augustus() first")
        return False

    print("\n⚡ Opening AUGUSTUS submission page...")
    main_handle = driver.current_window_handle
    
    driver.execute_script(f"window.open('{AUGUSTUS_URL}', '_blank');")
    time.sleep(3)
    driver.switch_to.window(driver.window_handles[-1])

    try:
        # Page load ke liye wait karo
        wait_ready(driver)
        time.sleep(3)

        # Check for any alert/popup from previous failed attempts
        try:
            alert = driver.switch_to.alert
            print(f"⚠️ Alert detected: {alert.text}")
            alert.accept()
            time.sleep(2)
        except:
            pass

        print("🔍 Locating sequence textarea...")
        textarea = WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.TAG_NAME, "textarea"))
        )

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", textarea)
        driver.execute_script("arguments[0].style.outline='3px solid #3b82f6';", textarea)

        print("⚡ Injecting sequence via JavaScript...")
        
        # JavaScript injection with proper events
        driver.execute_script("""
            arguments[0].value = arguments[1];
            arguments[0].dispatchEvent(new Event('input', { bubbles: true }));
            arguments[0].dispatchEvent(new Event('change', { bubbles: true }));
        """, textarea, fasta_sequence)

        time.sleep(2)

        # Verification
        actual_length = len(textarea.get_attribute("value"))
        expected_length = len(fasta_sequence)

        print(f"📊 Expected: {expected_length:,} chars")
        print(f"📊 Injected: {actual_length:,} chars")

        if actual_length == expected_length:
            print("✅ Sequence injected successfully.")
            
            # Check for server-side alert (sequence too large)
            time.sleep(2)
            try:
                alert = driver.switch_to.alert
                print(f"⚠️ Server alert: {alert.text}")
                alert.accept()
                print("❌ Server rejected sequence (too large)")
                driver.close()
                driver.switch_to.window(main_handle)
                return False
            except:
                pass
            
            # IMPORTANT FIX: Page ko re-initialize hone do
            print("⏳ Waiting for page to initialize after sequence paste...")
            time.sleep(3)
            
            # Scroll to top to ensure dropdown visible
            driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(1)
            
            return True
        else:
            print("⚠️ Length mismatch detected.")
            return False

    except Exception as e:
        print(f"❌ Injection error: {e}")
        traceback.print_exc()
        return False


def select_organism_arabidopsis_enhanced(driver):
    """
    Simplified version - sirf JavaScript fallback use karega jo working hai
    """
    try:
        print("🌱 Selecting organism → Arabidopsis thaliana…")
        
        # Page fully loaded hone ka wait karo
        wait_ready(driver)
        time.sleep(3)
        
        # DIRECT JAVASCRIPT FALLBACK - yeh working hai aapke log ke according
        print("   Using JavaScript direct selection...")
        driver.execute_script("""
            var select = document.querySelector('select[name="organism"]');
            if(select) {
                select.value = 'arabidopsis';
                select.dispatchEvent(new Event('change', { bubbles: true }));
                console.log('Arabidopsis selected via JS');
            } else {
                console.log('Select element not found');
            }
        """)
        time.sleep(2)
        
        # Verify selection
        try:
            select_elem = driver.find_element(By.XPATH, "//select[@name='organism']")
            selected_value = select_elem.get_attribute('value')
            if selected_value == 'arabidopsis':
                print("✅ Arabidopsis thaliana selected successfully.")
                return True
            else:
                print(f"⚠️ Selection may have failed (value: {selected_value})")
                return False
        except:
            print("⚠️ Could not verify selection")
            return False

    except Exception as e:
        print(f"❌ Error in select_organism: {e}")
        traceback.print_exc()
        return False


def run_augustus_enhanced(driver):
    """
    NON-BLOCKING SUBMISSION
    Prevents EdgeDriver timeout during long Augustus processing
    """

    try:
        print("🚀 Submitting AUGUSTUS job (non-blocking mode)…")

        wait_ready(driver)
        time.sleep(2)

        run_btn = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.XPATH, "//input[@type='submit' and @value='Run AUGUSTUS']")
            )
        )

        # Scroll into view
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", run_btn)
        time.sleep(1)

        # 🔥 CRITICAL FIX:
        # Instead of clicking button → directly submit the form
        driver.execute_script("""
            var btn = arguments[0];
            if(btn.form){
                btn.form.submit();
            }
        """, run_btn)

        print("✅ Form submitted successfully (driver will NOT block)")

        # IMPORTANT:
        # Do NOT wait for readyState here
        # Server now processing independently

        time.sleep(3)
        return True

    except Exception as e:
        print(f"❌ Submission failed: {e}")
        return False


def process_augustus_chunk(driver, fasta_sequence, species_name, chunk_begin, chunk_end, chunk_num, total_chunks, folders, is_positive):
    """
    Process a single chunk through Augustus - MODIFIED FOR FOLDER STRUCTURE
    is_positive parameter added
    """
    print(f"\n{'='*80}")
    print(f"📦 PROCESSING CHUNK {chunk_num}/{total_chunks}")
    print(f"   Coordinates: {chunk_begin:,} - {chunk_end:,}")
    print(f"{'='*80}")
    
    # Step 1: Paste sequence with verification
    success = paste_sequence_verified(driver, fasta_sequence)
    
    if not success:
        print(f"❌ Failed to paste sequence for chunk {chunk_num}")
        return False
    
    # Check driver connection after paste
    try:
        driver.current_url
    except:
        print("⚠️ Driver connection lost after paste")
        return False
    
    # Step 2: Wait for page to stabilize
    print("\n⏳ Waiting for page to fully initialize after sequence paste...")
    time.sleep(5)
    try:
        wait_ready(driver)
    except:
        print("⚠️ Page ready timeout, continuing anyway")
    
    # Step 3: Select organism with enhanced method
    print("\n" + "-"*40)
    success = select_organism_arabidopsis_enhanced(driver)
    if not success:
        print("❌ Failed to select organism")
        return False
    
    time.sleep(3)
    
    # Step 4: Run Augustus with enhanced method
    print("\n" + "-"*40)
    success = run_augustus_enhanced(driver)
    if not success:
        print("❌ Failed to run Augustus")
        return False
    
    # ================================================================
    # STEP 5: WAIT FOR "Done." TEXT (USING UNTITLED5 FUNCTION)
    # ================================================================
    completed = wait_for_augustus_done(
        driver, 
        folders, 
        chunk_num,
        max_wait_minutes=AUGUSTUS_COMPLETE_TIMEOUT
    )
    
    if not completed:
        print(f"❌ Chunk {chunk_num} failed - 'Done.' not detected within {AUGUSTUS_COMPLETE_TIMEOUT} minutes")
        return False
    
    # Give extra time for page to fully load
    print("⏳ Giving extra 5 seconds for page to stabilize...")
    time.sleep(5)
    wait_ready(driver)
    
    # ================================================================
    # STEP 6: Process results - MODIFIED FOR FOLDER STRUCTURE WITH APPENDING
    # ================================================================
    print("📊 Processing Augustus results...")
    aa_file, nt_file = handle_augustus_results(driver, species_name, chunk_begin, chunk_end, folders, chunk_num, is_positive)
    
    if aa_file or nt_file:
        print(f"✅ Chunk {chunk_num} results saved")
        return True
    else:
        print(f"⚠️ Chunk {chunk_num} results incomplete")
        return False


# ================================================================
# NEW FUNCTIONS ADDED FOR PARALLEL PROCESSING (ADD-ON)
# ================================================================

def launch_chunk_parallel(chunk_data, chunk_idx, species_name, folders, is_positive):
    """
    Single chunk launch karne ke liye - har chunk ka apna driver
    """
    chunk_seq, chunk_begin, chunk_end = chunk_data
    
    print(f"\n{'='*80}")
    print(f"🚀 LAUNCHING CHUNK {chunk_idx}")
    print(f"   Coordinates: {chunk_begin:,} - {chunk_end:,}")
    print(f"{'='*80}")
    
    # Create separate driver for this chunk - use Augustus-specific options
    aug_opts = get_augustus_driver_options()
    aug_driver = webdriver.Edge(service=Service(EDGEDRIVER_PATH), options=aug_opts)
    
    try:
        # Process chunk
        success = process_augustus_chunk(
            aug_driver,
            chunk_seq,
            species_name,
            chunk_begin,
            chunk_end,
            chunk_idx,
            999,  # Total chunks unknown, placeholder
            folders,
            is_positive
        )
        return (chunk_idx, success)
    finally:
        aug_driver.quit()


def process_augustus_parallel(fasta_sequence, species_name, begin, end, folders, is_positive):
    """
    PARALLEL PROCESSING - 2GB tak ke chunks ek saath launch karo (1 min gap)
    """
    print("\n" + "="*80)
    print(f"🚀 PARALLEL AUGUSTUS PROCESSING FOR {'POSITIVE' if is_positive else 'NEGATIVE'} CASE")
    print("="*80)

    # Step 1: Split sequence into chunks
    chunks = split_sequence_for_augustus(fasta_sequence, species_name, begin, end)
    
    if len(chunks) == 1:
        print("📦 Only one chunk - processing normally...")
        aug_opts = get_augustus_driver_options()
        aug_driver = webdriver.Edge(service=Service(EDGEDRIVER_PATH), options=aug_opts)
        try:
            success = process_augustus_chunk(
                aug_driver,
                chunks[0][0],
                species_name,
                chunks[0][1],
                chunks[0][2],
                1,
                1,
                folders,
                is_positive
            )
            return success
        finally:
            aug_driver.quit()
    
    print(f"\n📊 Total {len(chunks)} chunks to process")
    
    # Calculate total size of all chunks
    total_size_gb = 0
    chunk_sizes = []
    for chunk_seq, chunk_begin, chunk_end in chunks:
        chunk_size_mb = len(chunk_seq) / (1024 * 1024)
        chunk_size_gb = chunk_size_mb / 1024
        chunk_sizes.append(chunk_size_gb)
        total_size_gb += chunk_size_gb
    
    print(f"📊 Total size: {total_size_gb:.2f} GB")
    print(f"📊 2GB limit per batch")
    
    # Create batches (max 2GB per batch)
    batches = []
    current_batch = []
    current_batch_size = 0
    current_batch_indices = []
    
    for i, (chunk_seq, chunk_begin, chunk_end) in enumerate(chunks, 1):
        chunk_size_gb = chunk_sizes[i-1]
        
        if current_batch_size + chunk_size_gb <= 2.0:  # 2GB limit
            current_batch.append((chunk_seq, chunk_begin, chunk_end))
            current_batch_indices.append(i)
            current_batch_size += chunk_size_gb
        else:
            # Save current batch and start new one
            if current_batch:
                batches.append((current_batch, current_batch_indices, current_batch_size))
            current_batch = [(chunk_seq, chunk_begin, chunk_end)]
            current_batch_indices = [i]
            current_batch_size = chunk_size_gb
    
    # Add last batch
    if current_batch:
        batches.append((current_batch, current_batch_indices, current_batch_size))
    
    print(f"\n📦 Created {len(batches)} batches (each ≤ 2GB):")
    for batch_num, (batch, indices, size) in enumerate(batches, 1):
        print(f"   Batch {batch_num}: {len(batch)} chunks, {size:.2f} GB, Chunks: {indices}")
    
    # Process batches sequentially
    all_successful = True
    failed_chunks = []
    
    for batch_num, (batch, batch_indices, batch_size) in enumerate(batches, 1):
        print(f"\n{'='*80}")
        print(f"📦 PROCESSING BATCH {batch_num}/{len(batches)} ({batch_size:.2f} GB)")
        print(f"{'='*80}")
        
        print(f"\n⚡ Launching {len(batch)} chunks in parallel with 1-minute gap...")
        
        futures = []
        results = []
        
        # Launch all chunks in this batch with ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=len(batch)) as executor:
            
            for i, (chunk_data, chunk_idx) in enumerate(zip(batch, batch_indices)):
                # Submit chunk for processing
                future = executor.submit(
                    launch_chunk_parallel, 
                    chunk_data, chunk_idx, species_name, folders, is_positive
                )
                futures.append(future)
                
                print(f"   ✅ Launched chunk {chunk_idx} ({i+1}/{len(batch)})")
                
                if i < len(batch) - 1:
                    # Wait 1 minute before launching next chunk
                    print(f"   ⏱️ Waiting 60 seconds before chunk {batch_indices[i+1]}...")
                    time.sleep(60)
            
            print(f"\n📊 All {len(batch)} chunks launched! Waiting for results...")
            
            # Collect results as they complete
            for future in as_completed(futures):
                try:
                    chunk_idx, success = future.result(timeout=2700)  # 45 minutes timeout
                    
                    if success:
                        print(f"   ✅ Chunk {chunk_idx} completed successfully")
                        results.append((chunk_idx, True))
                    else:
                        print(f"   ❌ Chunk {chunk_idx} failed")
                        results.append((chunk_idx, False))
                        failed_chunks.append(chunk_idx)
                        
                except Exception as e:
                    print(f"   ❌ Error collecting chunk result: {e}")
                    failed_chunks.append("unknown")
        
        # Batch summary
        successful_batch = [r for r in results if r[1]]
        print(f"\n📊 BATCH {batch_num} SUMMARY:")
        print(f"   ✅ Successful: {len(successful_batch)}/{len(batch)}")
        if failed_chunks:
            print(f"   ❌ Failed in this batch: {len([f for f in failed_chunks if f in batch_indices])}")
        
        # Wait between batches
        if batch_num < len(batches):
            print(f"\n⏱️ Waiting 2 minutes before next batch...")
            time.sleep(120)
    
    # Final summary
    print(f"\n{'='*80}")
    print(f"📊 FINAL SUMMARY FOR {'POSITIVE' if is_positive else 'NEGATIVE'} CASE")
    print(f"{'='*80}")
    print(f"✅ Total chunks: {len(chunks)}")
    print(f"✅ Successful: {len(chunks) - len(failed_chunks)}")
    if failed_chunks:
        print(f"❌ Failed chunks: {len(failed_chunks)}")
        print(f"   Failed chunk numbers: {failed_chunks}")
    print(f"{'='*80}")
    
    return len(failed_chunks) == 0


# ================================================================
# MODIFIED FUNCTION: process_augustus_enhanced (UPDATED WITH PARALLEL)
# ================================================================

def process_augustus_enhanced(driver, fasta_sequence, species_name, begin, end, folders, is_positive):
    """
    Complete fixed Augustus processing pipeline with splitting - MODIFIED FOR PARALLEL PROCESSING
    is_positive parameter added
    """
    print("\n" + "="*80)
    print(f"🚀 ENHANCED AUGUSTUS PROCESSING PIPELINE (WITH PARALLEL EXECUTION)")
    print("="*80)

    # Delegate to parallel processor
    return process_augustus_parallel(fasta_sequence, species_name, begin, end, folders, is_positive)


def handle_augustus_results(driver, species_name, begin, end, folders, chunk_num=None, is_positive=True):
    """
    MODIFIED VERSION: Creates two separate files in respective folders:
    - protein/aa_positive/negative_species_begin_end.txt (amino acid sequences)
    - nucleotide/nt_positive/negative_species_begin_end.txt (coding sequences)
    
    If chunk_num is provided, results are appended to existing files
    All chunks for the same range go into the SAME file (append mode)
    """

    import re, time, os

    try:
        print("\n" + "="*80)
        print("📊 AUGUSTUS RESULTS HANDLER - DUAL FILE OUTPUT")
        print("="*80)

        safe_species = re.sub(r'\W+', '_', species_name)
        case_prefix = "positive" if is_positive else "negative"
        
        # File paths in respective folders - SAME FILE for all chunks of this range
        aa_file = os.path.join(folders["protein"], f"aa_{case_prefix}_{safe_species}_{begin}_{end}.txt")
        nt_file = os.path.join(folders["nucleotide"], f"nt_{case_prefix}_{safe_species}_{begin}_{end}.txt")
        
        # Check if this is first chunk or subsequent
        is_first_chunk = (chunk_num == 1 or chunk_num is None)
        
        print(f"📁 Protein results will be saved to: {aa_file}")
        print(f"📁 Nucleotide results will be saved to: {nt_file}")
        if chunk_num:
            print(f"📦 Processing chunk {chunk_num} - {'Creating new files' if is_first_chunk else 'Appending to existing files'}")

        # --------------------------------------------------
        # STEP 1: Wait for "graphical and text results"
        # --------------------------------------------------
        print("⏳ Waiting for 'graphical and text results' link...")

        results_link = WebDriverWait(driver, 900).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//a[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'graphical and text results')]"
            ))
        )

        print("✅ Results link found")

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", results_link)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", results_link)

        print("✅ Clicked 'graphical and text results'")
        time.sleep(5)

        wait_ready(driver)
        time.sleep(3)

        # ================ 🔥 2-MINUTE CHECK HERE ================
        print("\n" + "-"*60)
        print("🔍 CHECKING AUGUSTUS RESULTS PAGE (2 MINUTE TIMEOUT)")
        print("-"*60)
        
        # Wait for results to be ready (max 2 minutes) - USING UNTITLED5 FUNCTION
        results_ready = check_augustus_results_ready(driver, folders, chunk_num, max_wait_seconds=120)
        
        if not results_ready:
            print(f"❌ Chunk {chunk_num} failed - required links not found within 2 minutes")
            return None, None
        # ================ END OF 2-MINUTE CHECK ================

        # --------------------------------------------------
        # STEP 2: Click predicted amino acid sequences
        # --------------------------------------------------
        print("📥 Opening predicted amino acid sequences...")

        protein_link = WebDriverWait(driver, 60).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//a[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'predicted amino acid sequences')]"
            ))
        )

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", protein_link)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", protein_link)

        print("✅ Amino acid page loaded")
        time.sleep(5)

        wait_ready(driver)
        time.sleep(2)

        protein_fasta = ""
        try:
            pre = WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.TAG_NAME, "pre"))
            )
            protein_fasta = pre.text.strip()
            print(f"✅ Protein FASTA captured ({len(protein_fasta):,} chars)")
        except:
            print("⚠ Protein FASTA not found")

        # Go back to cabinet page
        driver.back()
        time.sleep(4)
        wait_ready(driver)

        # --------------------------------------------------
        # STEP 3: Click predicted coding sequences
        # --------------------------------------------------
        print("📥 Opening predicted coding sequences...")

        coding_link = WebDriverWait(driver, 60).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//a[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'predicted coding sequences')]"
            ))
        )

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", coding_link)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", coding_link)

        print("✅ Coding page loaded")
        time.sleep(5)

        wait_ready(driver)
        time.sleep(2)

        coding_fasta = ""
        try:
            pre = WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.TAG_NAME, "pre"))
            )
            coding_fasta = pre.text.strip()
            print(f"✅ Coding FASTA captured ({len(coding_fasta):,} chars)")
        except:
            print("⚠ Coding FASTA not found")

        # --------------------------------------------------
        # STEP 4: Save to separate files in respective folders - APPEND MODE for chunks
        # --------------------------------------------------
        
        # Save Protein (AA) file
        if protein_fasta:
            mode = 'w' if is_first_chunk else 'a'
            with open(aa_file, mode, encoding="utf-8") as f:
                if not is_first_chunk:
                    f.write("\n\n" + "="*60 + "\n")
                    f.write(f"CHUNK {chunk_num} RESULTS (Coordinates: {begin}-{end})\n")
                    f.write("="*60 + "\n\n")
                
                f.write(protein_fasta)
            
            print(f"✅ Protein results {'saved' if is_first_chunk else 'appended'} to: {aa_file}")
        else:
            print("⚠ No protein data to save")

        # Save Coding (NT) file
        if coding_fasta:
            mode = 'w' if is_first_chunk else 'a'
            with open(nt_file, mode, encoding="utf-8") as f:
                if not is_first_chunk:
                    f.write("\n\n" + "="*60 + "\n")
                    f.write(f"CHUNK {chunk_num} RESULTS (Coordinates: {begin}-{end})\n")
                    f.write("="*60 + "\n\n")
                
                f.write(coding_fasta)
            
            print(f"✅ Nucleotide results {'saved' if is_first_chunk else 'appended'} to: {nt_file}")
        else:
            print("⚠ No nucleotide data to save")

        print("="*80 + "\n")
        return aa_file, nt_file

    except Exception as e:
        print(f"❌ Error in handle_augustus_results: {e}")
        import traceback
        traceback.print_exc()
        return None, None


# ================== FASTA HEADER SHORTENING FUNCTION ==================
def shorten_fasta_header(fasta_sequence):
    """
    FASTA sequence ki header line ko shorten karega, lekin sequence ko affect nahi karega.
    """
    if not fasta_sequence:
        return fasta_sequence
    
    print("🔧 Processing FASTA header...")
    
    # Header alag karo aur base pairs alag
    header, base_pairs, bp_count = extract_sequence_only(fasta_sequence)
    
    if header:
        print(f"   Original header: {header[:100]}..." if len(header) > 100 else f"   Original header: {header}")
        
        # Shorten header if needed
        if len(header) > 100:
            shortened_header = header[:100] + "..."
            print(f"   Shortened header: {shortened_header}")
        else:
            shortened_header = header
            print(f"   Header already short enough")
        
        # Reconstruct FASTA with shortened header + base pairs
        shortened_fasta = shortened_header + '\n' + base_pairs
        
        print(f"   Base pairs: {bp_count:,} (unchanged)")
        print(f"   Original length: {len(fasta_sequence):,} chars")
        print(f"   Shortened length: {len(shortened_fasta):,} chars")
        
        return shortened_fasta
    
    return fasta_sequence

# ================== GET VALUES FUNCTIONS ==================

def get_negative_range_values(csv_path):
    """Get BEGIN and END from negative range CSV"""
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("❌ Negative range CSV empty")
            return None, None
        
        begin = df.iloc[0, 0]  # First column
        end = df.iloc[0, 1]    # Second column
        
        print(f"📌 NEGATIVE RANGE: BEGIN={begin:,}, END={end:,}")
        return begin, end
        
    except Exception as e:
        print(f"❌ Error reading negative range CSV: {e}")
        return None, None


def get_first_row_values(sorted_csv_path):
    """Sorted CSV me se first row ke Column A aur Column B uthao"""
    try:
        df = pd.read_csv(sorted_csv_path)
        if df.empty:
            print("❌ Sorted CSV empty hai")
            return None, None, None
        
        # FIX: Check if it's positive based on Column_O values, not filename
        # Check if all values in last column are positive
        col_O = df.iloc[:, -1] if df.shape[1] > 2 else pd.Series([1])  # Handle if only 2 columns
        
        # Convert to numeric and check
        col_O_numeric = pd.to_numeric(col_O, errors='coerce')
        is_positive = (col_O_numeric > 0).all() if not col_O_numeric.empty else True
        
        # For regular CSV, use the smallest as BEGIN, largest as END
        begin = df.iloc[0, 0]
        end = df.iloc[0, 1]
        
        if is_positive:
            print(f"📌 POSITIVE CASE: BEGIN={begin:,}, END={end:,}")
        else:
            print(f"📌 NEGATIVE/MIXED CASE: BEGIN={begin:,}, END={end:,}")
            
        return begin, end, is_positive
        
    except Exception as e:
        print(f"❌ Error reading first row: {e}")
        return None, None, None

# ================== PROCESS CSV FUNCTION - MODIFIED FOR FOLDER STRUCTURE ==================

def process_csv(species_name, driver, folders):
    """
    Fully automated CSV processing -> GenBank -> Download Full FASTA -> AUGUSTUS.
    MODIFIED FOR FOLDER STRUCTURE - FIXED NEGATIVE CASE RANGE EXTRACTION
    """
    try:
        safe_name = species_name.replace(" ", "_") + ".csv"
        input_path = os.path.join(folders["excel"], safe_name)

        if not os.path.exists(input_path):
            print(f"❌ CSV not found: {input_path}")
            return None, None

        df = pd.read_csv(input_path)

        # Column I (9th, zero-based = 8)
        col_I = df.iloc[:, 8]
        # Column J (10th, zero-based = 9)
        col_J = df.iloc[:, 9]
        # Column O (last column) - CONVERT TO NUMERIC TO HANDLE ALL CASES
        col_O = pd.to_numeric(df.iloc[:, -1], errors='coerce')

        # DEBUG: Print what's in Column O
        print(f"📊 DEBUG: Column_O values: {col_O.tolist()}")
        print(f"📊 DEBUG: Column_O data types: {col_O.dtype}")
        print(f"📊 DEBUG: Column_O unique values: {col_O.unique().tolist()}")

        # FIXED CONDITION CHECK: Handle NaN values properly
        has_positive = any(col_O > 0)
        has_negative = any(col_O < 0)

        # Additional check for zero values
        has_zero = any(col_O == 0)
        
        print(f"📊 DEBUG: has_positive={has_positive}, has_negative={has_negative}, has_zero={has_zero}")

        sorted_paths = []

        # Case 1: Only positive (OR zero/positive)
        if has_positive and not has_negative:
            # POSITIVE CASE: Use as is
            # FIX: Ensure we're working with valid numbers
            col_I_valid = col_I.dropna()
            col_J_valid = col_J.dropna()
            col_O_valid = col_O.dropna()
            
            if len(col_I_valid) == 0 or len(col_J_valid) == 0:
                print("❌ No valid coordinates found after dropping NaN")
                return None, None
            
            col_A_sorted = sorted(col_I_valid)
            col_B_sorted = sorted(col_J_valid, reverse=True)
            
            # Make sure we have matching lengths
            min_len = min(len(col_A_sorted), len(col_B_sorted), len(col_O_valid))
            col_A_sorted = col_A_sorted[:min_len]
            col_B_sorted = col_B_sorted[:min_len]
            col_O_sorted = col_O_valid.tolist()[:min_len]
            
            sub_df = pd.DataFrame({
                "Column_A": col_A_sorted,
                "Column_B": col_B_sorted,
                "Column_O_original": col_O_sorted
            })
            sorted_name = species_name.replace(" ", "_") + "_sorted.csv"
            sorted_path = os.path.join(folders["excel"], sorted_name)
            
            # Check if file is open
            if os.path.exists(sorted_path):
                try:
                    os.remove(sorted_path)
                except:
                    pass
            
            sub_df.to_csv(sorted_path, index=False)
            sorted_paths.append(sorted_path)
            print(f"✅ Sorted CSV saved: {sorted_path}")
            print("📊 Sorted Excel Sheet is POSITIVE")

        # ========== FIXED NEGATIVE CASE - JAGAH 1 ==========
        # Case 2: Only negative - DONO COLUMNS ALAG-ALAG SORT
        elif has_negative and not has_positive:
            print("🔍 Processing NEGATIVE case...")
            
            col_I_valid = col_I.dropna()
            col_J_valid = col_J.dropna()
            
            if len(col_I_valid) == 0 or len(col_J_valid) == 0:
                print("❌ No valid coordinates found in negative case")
                return None, None
            
            # DONO COLUMNS ALAG-ALAG SORT KARO
            # Column_A: largest → smallest (descending)
            # Column_B: smallest → largest (ascending)
            col_A_sorted = sorted(col_I_valid, reverse=True)  # largest → smallest
            col_B_sorted = sorted(col_J_valid)                 # smallest → largest
            
            # Make sure we have matching lengths
            min_len = min(len(col_A_sorted), len(col_B_sorted))
            col_A_sorted = col_A_sorted[:min_len]
            col_B_sorted = col_B_sorted[:min_len]
            
            neg_df_sorted = pd.DataFrame({
                "Column_A": col_A_sorted,
                "Column_B": col_B_sorted
            })
            
            sorted_name = species_name.replace(" ", "_") + "_negative_sorted.csv"
            sorted_path = os.path.join(folders["excel"], sorted_name)
            
            # Check if file is open
            if os.path.exists(sorted_path):
                try:
                    os.remove(sorted_path)
                except:
                    pass
            
            neg_df_sorted.to_csv(sorted_path, index=False)
            sorted_paths.append(sorted_path)
            print(f"✅ Negative sorted CSV saved with {len(neg_df_sorted)} rows")
            
            # Display first few rows to verify sorting
            print("\n📊 First 5 rows of sorted negative CSV (Column_A descending, Column_B ascending):")
            print(neg_df_sorted.head().to_string())

        # Case 3: Mixed (positive + negative)
        else:
            print("🔍 Processing MIXED case (positive + negative)...")
            
            # FIX: Handle NaN values in the DataFrame
            df_clean = df.copy()
            df_clean['temp_O'] = pd.to_numeric(df.iloc[:, -1], errors='coerce')
            
            # Positive rows
            df_pos = df_clean[df_clean['temp_O'] > 0]
            if not df_pos.empty:
                # Ensure we have valid coordinates
                pos_coords_I = df_pos.iloc[:, 8].dropna()
                pos_coords_J = df_pos.iloc[:, 9].dropna()
                
                if len(pos_coords_I) > 0 and len(pos_coords_J) > 0:
                    col_A_sorted_pos = sorted(pos_coords_I)
                    col_B_sorted_pos = sorted(pos_coords_J, reverse=True)
                    
                    # Match lengths
                    min_len = min(len(col_A_sorted_pos), len(col_B_sorted_pos))
                    col_A_sorted_pos = col_A_sorted_pos[:min_len]
                    col_B_sorted_pos = col_B_sorted_pos[:min_len]
                    
                    pos_df = pd.DataFrame({
                        "Column_A": col_A_sorted_pos,
                        "Column_B": col_B_sorted_pos,
                        "Column_O_original": df_pos['temp_O'].tolist()[:min_len]
                    })
                    pos_name = species_name.replace(" ", "_") + "_positive_sorted.csv"
                    pos_path = os.path.join(folders["excel"], pos_name)
                    
                    # Check if file is open
                    if os.path.exists(pos_path):
                        try:
                            os.remove(pos_path)
                        except:
                            pass
                    
                    pos_df.to_csv(pos_path, index=False)
                    sorted_paths.append(pos_path)
                    print(f"✅ Positive Sorted CSV saved: {pos_path}")
                else:
                    print("⚠️ Positive rows found but no valid coordinates")
            else:
                print("⚠️ No positive rows found")

            # ========== FIXED NEGATIVE CASE - JAGAH 2 ==========
            # Negative rows - DONO COLUMNS ALAG-ALAG SORT
            df_neg = df_clean[df_clean['temp_O'] < 0]
            if not df_neg.empty:
                print(f"📊 Processing {len(df_neg)} negative alignments...")
                
                col_A_neg = df_neg.iloc[:, 8].dropna().tolist()
                col_B_neg = df_neg.iloc[:, 9].dropna().tolist()
                
                if col_A_neg and col_B_neg:
                    # DONO COLUMNS ALAG-ALAG SORT KARO
                    # Column_A: largest → smallest (descending)
                    # Column_B: smallest → largest (ascending)
                    col_A_sorted_neg = sorted(col_A_neg, reverse=True)  # largest → smallest
                    col_B_sorted_neg = sorted(col_B_neg)                 # smallest → largest
                    
                    # Match lengths
                    min_len_neg = min(len(col_A_sorted_neg), len(col_B_sorted_neg))
                    col_A_sorted_neg = col_A_sorted_neg[:min_len_neg]
                    col_B_sorted_neg = col_B_sorted_neg[:min_len_neg]
                    
                    neg_df_sorted = pd.DataFrame({
                        "Column_A": col_A_sorted_neg,
                        "Column_B": col_B_sorted_neg
                    })
                    
                    neg_name = species_name.replace(" ", "_") + "_negative_sorted.csv"
                    neg_path = os.path.join(folders["excel"], neg_name)
                    
                    # Check if file is open
                    if os.path.exists(neg_path):
                        try:
                            os.remove(neg_path)
                        except:
                            pass
                    
                    neg_df_sorted.to_csv(neg_path, index=False)
                    sorted_paths.append(neg_path)
                    print(f"✅ Negative sorted CSV saved with {len(neg_df_sorted)} rows")
                    print("\n📊 First 5 rows of sorted negative CSV (mixed case, Column_A descending, Column_B ascending):")
                    print(neg_df_sorted.head().to_string())
                else:
                    print("⚠️ Negative rows found but no valid coordinates")
            else:
                print("⚠️ No negative rows found")
                    
        # If no sorted paths were created, return None
        if not sorted_paths:
            print("❌ No valid data to process in CSV")
            return None, None

        # Process each sorted CSV with FIXED NEGATIVE CASE LOGIC
        for spath in sorted_paths:
            filename = os.path.basename(spath).lower()
            is_negative_file = filename.endswith("_negative_sorted.csv")
        
            if is_negative_file:
                # ========== FIXED NEGATIVE CASE RANGE EXTRACTION ==========
                # For negative case, we need:
                # BEGIN = min of Column_B (smallest value in second column)
                # END = max of Column_A (largest value in first column)
                
                df_neg_current = pd.read_csv(spath)
                
                # Print debug info
                print(f"\n📊 DEBUG - Negative CSV columns: {df_neg_current.columns.tolist()}")
                
                # Check if we have the expected columns
                if 'Column_A' in df_neg_current.columns and 'Column_B' in df_neg_current.columns:
                    begin = df_neg_current["Column_B"].min()  # FIX: min of Column_B for BEGIN
                    end = df_neg_current["Column_A"].max()    # FIX: max of Column_A for END
                else:
                    # Fallback for older format
                    begin = df_neg_current.iloc[:, 1].min()   # Second column min for BEGIN
                    end = df_neg_current.iloc[:, 0].max()     # First column max for END
                
                is_positive = False
        
                print(f"\n{'='*80}")
                print(f"📊 PROCESSING NEGATIVE CASE: {os.path.basename(spath)}")
                print(f"{'='*80}")
                print(f"📌 NEGATIVE CASE: BEGIN from Column_B min = {begin:,}")
                print(f"📌 NEGATIVE CASE: END from Column_A max = {end:,}")
                print(f"📌 NEGATIVE CASE: Total range = {end - begin + 1:,} bp")
        
            else:
                # For positive files, use the original logic
                begin, end, is_positive = get_first_row_values(spath)
        
                print(f"\n{'='*80}")
                print(f"📊 PROCESSING POSITIVE CASE: {os.path.basename(spath)}")
                print(f"{'='*80}")
                
            if begin is None or end is None:
                print(f"⚠️ Skipping {spath} - could not get begin/end values")
                continue

            # CRITICAL VALIDATION
            print(f"📈 COORDINATE VALIDATION:")
            print(f"   BEGIN: {begin:,}")
            print(f"   END: {end:,}")
            
            if begin > end:
                print("⚠️ Coordinate order unexpected, correcting...")
                begin, end = end, begin
                        
            original_length = end - begin + 1
            print(f"   Original length: {original_length:,} bp")
            
            # Apply adjustment based on case
            if is_positive:
                adjusted_begin = max(1, begin - 1000)
                adjusted_end = end + 1000
                print(f"🔄 POSITIVE ADJUSTMENT:")
                print(f"   BEGIN {begin:,} → {adjusted_begin:,} (-1000)")
                print(f"   END {end:,} → {adjusted_end:,} (+1000)")
            
            else:
                # NEGATIVE CASE - same adjustment logic
                adjusted_begin = max(1, begin - 1000)
                adjusted_end = end + 1000
            
                print(f"🔄 NEGATIVE ADJUSTMENT:")
                print(f"   BEGIN {begin:,} → {adjusted_begin:,} (-1000)")
                print(f"   END {end:,} → {adjusted_end:,} (+1000)")
            
            adjusted_length = adjusted_end - adjusted_begin + 1
            print(f"📊 FINAL REGION:")
            print(f"   Coordinates: {adjusted_begin:,} - {adjusted_end:,}")
            print(f"   Length: {adjusted_length:,} bp")
            print(f"   Buffer added: {adjusted_length - original_length:,} bp")
            
            # Update values
            begin = adjusted_begin
            end = adjusted_end

            main_handle = driver.current_window_handle
            
            # GenBank Interaction
            try:
                genbank_link = WebDriverWait(driver, TIMEOUT).until(
                    EC.element_to_be_clickable((By.LINK_TEXT, "GenBank"))
                )
                highlight_and_click(driver, genbank_link)
                print("✅ GenBank link clicked.")

                WebDriverWait(driver, 10).until(lambda d: len(d.window_handles) > 1)
                
                # Switch to new tab (should be GenBank)
                for handle in driver.window_handles:
                    if handle != main_handle:
                        driver.switch_to.window(handle)
                        break
                
                wait_ready(driver)
                print("🔄 Switched to GenBank tab.")

            except Exception as e:
                print(f"❌ Could not click GenBank link: {e}")
                continue

            # ====================== DEBUGGING FOR WINDOW MANAGEMENT ======================
            print("\n" + "="*80)
            print("🔍 DEBUGGING WINDOW MANAGEMENT ISSUE")
            print("="*80)
            
            # Wait for GenBank page to load completely
            print("\n📊 DEBUG 1: WAITING FOR GENBANK PAGE TO LOAD")
            try:
                wait_ready(driver)
                time.sleep(5)
                print(f"   Current URL after GenBank click: {driver.current_url}")
                print(f"   Page title: {driver.title}")
                
                # Check if we're on correct GenBank page
                if "ncbi.nlm.nih.gov" not in driver.current_url:
                    print(f"   ⚠️ Warning: May not be on GenBank page")
                    print(f"   Actual page: {driver.current_url}")
                    
                    # Try to find and click GenBank link again
                    try:
                        print(f"   Trying to find GenBank link on current page")
                        genbank_links = driver.find_elements(By.PARTIAL_LINK_TEXT, "GenBank")
                        if genbank_links:
                            print(f"   Found {len(genbank_links)} GenBank links, clicking first")
                            genbank_links[0].click()
                            time.sleep(5)
                            wait_ready(driver)
                    except:
                        pass
            except Exception as e:
                print(f"   ❌ Error waiting for GenBank page: {e}")
            
            # ====================== CRITICAL FIX: CHECK AND SWITCH TO CORRECT WINDOW ======================
            print("\n📊 DEBUG 2: CHECKING WINDOW HANDLES")
            try:
                all_handles = driver.window_handles
                print(f"   Total window handles: {len(all_handles)}")
                print(f"   Current handle: {driver.current_window_handle}")
                
                # Find the GenBank window (not downloads window)
                genbank_handle = None
                downloads_handle = None
                
                for handle in all_handles:
                    driver.switch_to.window(handle)
                    current_url = driver.current_url
                    print(f"   Handle {handle}: {current_url[:100]}")
                    
                    if "edge://downloads" in current_url or "edge://" in current_url:
                        downloads_handle = handle
                        print(f"   ⚠️ Found downloads window")
                    elif "ncbi.nlm.nih.gov" in current_url:
                        genbank_handle = handle
                        print(f"   ✓ Found GenBank window")
                
                # CRITICAL: Switch to GenBank window if we're on downloads
                if driver.current_window_handle == downloads_handle and genbank_handle:
                    print(f"   ⚠️ Currently on downloads window, switching to GenBank window")
                    driver.switch_to.window(genbank_handle)
                    print(f"   Switched to GenBank window")
                    time.sleep(3)
                    wait_ready(driver)
                
                print(f"   Final URL after switching: {driver.current_url[:100]}...")
                print(f"   Final page title: {driver.title}")
                
            except Exception as e:
                print(f"   ❌ Error checking windows: {e}")
                traceback.print_exc()
            
            # ====================== DEBUGGING FOR FASTA LINK ======================
            print("\n📊 DEBUG 3: LOOKING FOR FASTA LINK WITH MULTIPLE METHODS")
            
            fasta_element = None
            fasta_method = ""
            max_attempts = 3
            
            for attempt in range(max_attempts):
                print(f"\n   Attempt {attempt + 1}/{max_attempts}")
                
                try:
                    # Method 1: Try LINK_TEXT
                    try:
                        print("      Trying LINK_TEXT: 'FASTA'")
                        fasta_element = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.LINK_TEXT, "FASTA"))
                        )
                        fasta_method = "LINK_TEXT"
                        print(f"      ✓ Found FASTA via LINK_TEXT")
                        break
                    except:
                        print(f"      ✗ LINK_TEXT failed")
                    
                    # Method 2: Try PARTIAL_LINK_TEXT
                    try:
                        print("      Trying PARTIAL_LINK_TEXT: 'FASTA'")
                        fasta_element = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT, "FASTA"))
                        )
                        fasta_method = "PARTIAL_LINK_TEXT"
                        print(f"      ✓ Found FASTA via PARTIAL_LINK_TEXT")
                        break
                    except:
                        print(f"      ✗ PARTIAL_LINK_TEXT failed")
                    
                    # Method 3: Try XPATH with href containing fasta
                    try:
                        print("      Trying XPATH: //a[contains(@href, 'fasta')]")
                        fasta_element = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.XPATH, "//a[contains(@href, 'fasta') or contains(@href, 'FASTA')]"))
                        )
                        fasta_method = "XPATH href contains fasta"
                        print(f"      ✓ Found FASTA via XPATH href")
                        break
                    except:
                        print(f"      ✗ XPATH href failed")
                    
                    # Method 4: Try XPATH with text containing FASTA
                    try:
                        print("      Trying XPATH: //a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'fasta')]")
                        fasta_element = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.XPATH, "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'fasta')]"))
                        )
                        fasta_method = "XPATH text contains fasta"
                        print(f"      ✓ Found FASTA via XPATH text")
                        break
                    except:
                        print(f"      ✗ XPATH text failed")
                    
                    # Method 5: Try to find format options
                    try:
                        print("      Looking for format options")
                        # Look for display format section
                        format_sections = driver.find_elements(By.XPATH, "//*[contains(text(), 'Display') or contains(text(), 'Format') or contains(text(), 'View')]")
                        if format_sections:
                            print(f"      Found {len(format_sections)} format/display sections")
                            # Try to find FASTA in these sections
                            for section in format_sections:
                                try:
                                    fasta_in_section = section.find_element(By.XPATH, ".//a[contains(., 'FASTA')]")
                                    fasta_element = fasta_in_section
                                    fasta_method = "Format section"
                                    print(f"      ✓ Found FASTA in format section")
                                    break
                                except:
                                    continue
                            if fasta_element:
                                break
                    except:
                        print(f"      ✗ Format search failed")
                    
                    # If not found, wait and retry
                    if attempt < max_attempts - 1:
                        print(f"      ⏳ Waiting 2 seconds before next attempt...")
                        time.sleep(2)
                        
                except Exception as e:
                    print(f"      ❌ Attempt {attempt + 1} failed: {str(e)[:100]}")
                    if attempt < max_attempts - 1:
                        time.sleep(2)
            
            # ====================== ELEMENT PROPERTIES DEBUGGING ======================
            if fasta_element:
                print(f"\n📊 DEBUG 4: FASTA ELEMENT PROPERTIES")
                try:
                    print(f"   Found via: {fasta_method}")
                    print(f"   Element text: '{fasta_element.text}'")
                    print(f"   Element href: {fasta_element.get_attribute('href')[:100]}...")
                    print(f"   Is displayed: {fasta_element.is_displayed()}")
                    print(f"   Is enabled: {fasta_element.is_enabled()}")
                    
                    # Highlight element
                    driver.execute_script("arguments[0].style.border='3px solid red'; arguments[0].style.backgroundColor='yellow';", fasta_element)
                    time.sleep(1)
                    
                except Exception as e:
                    print(f"   ⚠️ Could not get all properties: {e}")
            else:
                print(f"\n❌ FASTA LINK NOT FOUND AFTER {max_attempts} ATTEMPTS")
                
                # Try alternative approach: Direct URL construction
                print(f"\n📊 DEBUG 5: TRYING DIRECT URL APPROACH")
                current_url = driver.current_url
                print(f"   Current URL: {current_url[:100]}...")
                
                if "report=genbank" in current_url:
                    fasta_url = current_url.replace("report=genbank", "report=fasta")
                    print(f"   Constructed FASTA URL: {fasta_url[:100]}...")
                    
                    try:
                        driver.get(fasta_url)
                        wait_ready(driver)
                        time.sleep(5)
                        print(f"   ✓ Loaded FASTA URL directly")
                        
                        # Now try to find FASTA content
                        try:
                            fasta_pre = WebDriverWait(driver, 10).until(
                                EC.presence_of_element_located((By.CSS_SELECTOR, "pre"))
                            )
                            fasta_sequence = fasta_pre.text
                            if fasta_sequence and len(fasta_sequence) > 100:
                                print(f"   ✓ FASTA sequence captured directly from URL")
                                print("✅ FASTA link clicked via direct URL.")
                                # Skip to FASTA capture section
                                # We'll handle the rest after this debugging section
                            else:
                                print(f"   ✗ FASTA sequence too short or not found")
                                continue
                        except:
                            print(f"   ✗ Could not get FASTA content from direct URL")
                            continue
                    except Exception as e:
                        print(f"   ❌ Direct URL failed: {e}")
                        continue
                else:
                    print(f"   ❌ Cannot construct FASTA URL from current URL")
                    
                    # Save page for debugging
                    try:
                        debug_file = os.path.join(folders["base"], f"no_fasta_page_{int(time.time())}.html")
                        with open(debug_file, "w", encoding="utf-8") as f:
                            f.write(driver.page_source)
                        print(f"   💾 Page source saved: {debug_file}")
                    except:
                        pass
                    
                    continue
            
            # ====================== CLICKING FASTA LINK ======================
            print(f"\n📊 DEBUG 6: ATTEMPTING TO CLICK FASTA LINK")
            try:
                if fasta_element:
                    print(f"   Clicking FASTA element...")
                    
                    # Scroll to element
                    driver.execute_script("arguments[0].scrollIntoView({block:'center', behavior:'smooth'});", fasta_element)
                    time.sleep(2)
                    
                    # Try multiple click methods
                    clicked = False
                    
                    # Method 1: Regular click
                    try:
                        fasta_element.click()
                        clicked = True
                        print(f"   ✓ Regular click successful")
                    except:
                        print(f"   ✗ Regular click failed, trying JS click")
                        
                        # Method 2: JavaScript click
                        try:
                            driver.execute_script("arguments[0].click();", fasta_element)
                            clicked = True
                            print(f"   ✓ JS click successful")
                        except:
                            print(f"   ✗ JS click failed, trying action chain")
                            
                            # Method 3: Action chains
                            try:
                                actions = ActionChains(driver)
                                actions.move_to_element(fasta_element).click().perform()
                                clicked = True
                                print(f"   ✓ Action chain click successful")
                            except:
                                print(f"   ✗ All click methods failed")
                    
                    if clicked:
                        print(f"   ✅ FASTA link clicked successfully")
                        time.sleep(5)
                        
                        # Check if new window opened
                        current_handles = driver.window_handles
                        if len(current_handles) > len(all_handles):
                            print(f"   New window opened, switching to it")
                            # Switch to the newest window
                            for handle in current_handles:
                                if handle not in all_handles:
                                    driver.switch_to.window(handle)
                                    break
                        else:
                            print(f"   Still in same window")
                    else:
                        print(f"   ❌ Could not click FASTA element")
                        continue
                        
                else:
                    print(f"   ❌ No FASTA element to click")
                    continue
                    
            except Exception as e:
                print(f"   ❌ Error clicking FASTA: {e}")
                traceback.print_exc()
                continue
            
            print("="*80 + "\n")
            # ====================== END OF DEBUGGING ======================
                
            # Sidebar shutter
            try:
                shutter_btn = WebDriverWait(driver, TIMEOUT).until(
                    EC.element_to_be_clickable((By.ID, "EntrezSystem2.PEntrez.Nuccore.Sequence_ResultsPanel.Sequence_SingleItemSupl.Sequence_ViewerGenbankSidePanel.Sequence_ViewerChangeRegion.Shutter"))
                )
                highlight_and_click(driver, shutter_btn)
                print("✅ Sidebar shutter opened.")
                time.sleep(2)
            except Exception as e:
                print(f"❌ Could not open sidebar shutter: {e}")
                try:
                    # Try alternative ID
                    shutter_btn = driver.find_element(By.XPATH, "//a[contains(text(), 'Change region') or contains(text(), 'Show sidebar')]")
                    highlight_and_click(driver, shutter_btn)
                    print("✅ Sidebar opened via alternative method.")
                    time.sleep(2)
                except:
                    print(f"⚠️ Could not open sidebar, continuing anyway")
            
            # Selected region radio button
            try:
                sel_region_btn = WebDriverWait(driver, TIMEOUT).until(
                    EC.element_to_be_clickable((By.ID, "crselregion"))
                )
                highlight_and_click(driver, sel_region_btn)
                print("✅ Selected region button clicked.")
                time.sleep(1)
            except Exception as e:
                print(f"❌ Could not click Selected region button: {e}")
                try:
                    sel_region_btn = driver.find_element(By.XPATH, "//input[@type='radio' and contains(@value, 'sel')]")
                    highlight_and_click(driver, sel_region_btn)
                    print("✅ Selected region button clicked via XPath.")
                    time.sleep(1)
                except:
                    print(f"⚠️ Could not select region, continuing anyway")

            # Fill Begin and End values
            try:
                begin_box = WebDriverWait(driver, TIMEOUT).until(
                    EC.presence_of_element_located((By.ID, "crfrom"))
                )
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", begin_box)
                begin_box.clear()
                begin_box.send_keys(str(begin))
                print(f"✅ Begin value filled: {begin}")
                time.sleep(0.5)
            except Exception as e:
                print(f"❌ Could not fill Begin: {e}")
                try:
                    begin_box = driver.find_element(By.XPATH, "//input[contains(@id, 'from') or contains(@name, 'from')]")
                    begin_box.clear()
                    begin_box.send_keys(str(begin))
                    print(f"✅ Begin value filled via XPath: {begin}")
                    time.sleep(0.5)
                except:
                    print(f"⚠️ Could not fill Begin value")
                    continue

            try:
                end_box = WebDriverWait(driver, TIMEOUT).until(
                    EC.presence_of_element_located((By.ID, "crto"))
                )
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", end_box)
                end_box.clear()
                end_box.send_keys(str(end))
                print(f"✅ End value filled: {end}")
                time.sleep(0.5)
            except Exception as e:
                print(f"❌ Could not fill End: {e}")
                try:
                    end_box = driver.find_element(By.XPATH, "//input[contains(@id, 'to') or contains(@name, 'to')]")
                    end_box.clear()
                    end_box.send_keys(str(end))
                    print(f"✅ End value filled via XPath: {end}")
                    time.sleep(0.5)
                except:
                    print(f"⚠️ Could not fill End value")
                    continue

            # Update view button
            try:
                update_btn = WebDriverWait(driver, TIMEOUT).until(
                    EC.element_to_be_clickable((By.ID, "updateselregion"))
                )
                highlight_and_click(driver, update_btn)
                print("✅ Update View button clicked.")
                time.sleep(3)  # Wait for update to complete
            except Exception as e:
                print(f"❌ Could not click Update View: {e}")
                try:
                    update_btn = driver.find_element(By.XPATH, "//input[@type='button' and contains(@value, 'Update')]")
                    highlight_and_click(driver, update_btn)
                    print("✅ Update button clicked via XPath.")
                    time.sleep(3)
                except:
                    print(f"⚠️ Could not update view, trying to continue")

            # ====================== HTML CONFIRMATION TEXT CHECK ======================
            print("\n" + "="*80)
            print("🔍 CHECKING FOR REGION CONFIRMATION TEXT")
            print("="*80)

            # Wait for the confirmation text to appear after Update View click
            # Text format: "Showing X.XXMb region from base Y to Z"
            confirmation_xpath = "//span[contains(@class, 'icon') and contains(text(), 'Showing') and contains(text(), 'region from base')]"

            try:
                # Wait up to 30 seconds for the text to appear
                confirmation_element = WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.XPATH, confirmation_xpath))
                )
                
                confirmation_text = confirmation_element.text
                print(f"✅ Region confirmation detected: {confirmation_text}")
                
                # Extract values from the text for verification (optional)
                match = re.search(r'Showing ([\d.]+)Mb region from base (\d+) to (\d+)', confirmation_text)
                if match:
                    size_mb = match.group(1)
                    shown_begin = match.group(2)
                    shown_end = match.group(3)
                    print(f"   📊 Size: {size_mb} MB")
                    print(f"   📍 Range: {shown_begin} - {shown_end}")
                    print(f"   ✅ Matches expected range: {shown_begin == str(begin) and shown_end == str(end)}")
                
                # Highlight the element like FASTA and Alignments buttons
                driver.execute_script("arguments[0].style.outline='3px solid #22c55e'; arguments[0].style.backgroundColor='#f0fdf4';", confirmation_element)
                print("✅ HTML confirmation text recognized and highlighted")
                
                # ====================== WAIT 15 SECONDS AND CLICK CANCEL BUTTON ======================
                print(f"\n⏳ Waiting 15 seconds before clicking Cancel button...")
                time.sleep(15)
                
                # Find and click Cancel button using JavaScript
                try:
                    # Cancel button with image and text
                    cancel_xpath = "//button[img[@alt='Cancel image'] and contains(text(), 'Cancel')]"
                    cancel_button = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, cancel_xpath))
                    )
                    
                    # Highlight Cancel button
                    driver.execute_script("arguments[0].style.outline='3px solid #ff0000'; arguments[0].style.backgroundColor='#ffeeee';", cancel_button)
                    print("✅ Cancel button found and highlighted")
                    
                    # Click using JavaScript
                    driver.execute_script("arguments[0].click();", cancel_button)
                    print("✅ Cancel button clicked using JavaScript")
                    
                    # Wait a moment for cancel to take effect
                    time.sleep(2)
                    
                except Exception as cancel_error:
                    print(f"⚠️ Could not click Cancel button: {cancel_error}")
                    # Try alternative selector
                    try:
                        cancel_alt = driver.find_element(By.XPATH, "//button[contains(text(), 'Cancel')]")
                        driver.execute_script("arguments[0].click();", cancel_alt)
                        print("✅ Cancel button clicked via alternative selector")
                        time.sleep(2)
                    except:
                        print("⚠️ Cancel button not found, continuing anyway")
                
            except Exception as e:
                print(f"⚠️ Could not detect confirmation text: {e}")
                print("⚠️ Continuing with download process...")

            print("="*80 + "\n")
            # ====================== END OF CONFIRMATION AND CANCEL ======================
            
            # ====================== DOWNLOAD FULL FASTA USING "SEND TO" ======================
            print("\n" + "="*80)
            print("📥 DOWNLOADING FULL SEQUENCE VIA SEND TO")
            print("="*80)
            
            # Download full FASTA using "Send to" -> "File" -> "Create File"
            fasta_file_path = download_full_fasta_sequence(driver, species_name, begin, end, folders, is_positive)
            
            if not fasta_file_path:
                print("❌ Failed to download FASTA file")
                driver.close()
                driver.switch_to.window(main_handle)
                continue
            
            # ================================================================
            # CHANGED: Read the downloaded FASTA file - SIRF BASE PAIRS RETURN HOGA
            # ================================================================
            base_pairs_only = read_fasta_file(fasta_file_path)
            
            if not base_pairs_only:
                print("❌ Failed to read downloaded FASTA file")
                driver.close()
                driver.switch_to.window(main_handle)
                continue
            
            # Extract base pairs count
            total_bp = len(base_pairs_only)
            print(f"✅ FULL SEQUENCE downloaded for {species_name} ({begin}-{end}), base pairs={total_bp:,}")
            
            # ================================================================
            # NEW: Create FASTA with custom shortened header
            # ================================================================
            # Custom header banate hain - species name aur range ke saath
            case_prefix = "positive" if is_positive else "negative"
            custom_header = f">{case_prefix}_{species_name}_{begin}_{end}"
            fasta_sequence = custom_header + '\n' + base_pairs_only
            
            print(f"\n📌 Custom header created: {custom_header}")
            
            # ====================== SHORTEN FASTA HEADER ======================
            print("\n🔧 Shortening FASTA header for Augustus...")
            shortened_fasta = shorten_fasta_header(fasta_sequence)
            
            if shortened_fasta != fasta_sequence:
                print("✅ FASTA header successfully shortened")
                # Save the shortened FASTA for debugging
                try:
                    debug_fasta_path = os.path.join(folders["fasta"], f"shortened_{int(time.time())}.txt")
                    with open(debug_fasta_path, "w", encoding="utf-8") as f:
                        f.write(shortened_fasta)
                    print(f"📝 Shortened FASTA saved: {debug_fasta_path}")
                except:
                    pass
                # Use shortened FASTA for Augustus
                fasta_for_augustus = shortened_fasta
            else:
                print("⚠️ FASTA header not changed (already short or pattern not recognized)")
                fasta_for_augustus = fasta_sequence

            # ====================== ENHANCED AUGUSTUS PROCESSING WITH PARALLEL EXECUTION ======================
            if fasta_for_augustus:
                print("🚀 Starting enhanced Augustus processing with splitting (using separate driver for each chunk)...")
            
                # 🔥 USE AUGUSTUS-SPECIFIC DRIVER OPTIONS
                success = process_augustus_enhanced(
                    driver,  # Main driver passed but won't be used directly
                    fasta_for_augustus,
                    species_name,
                    begin,
                    end,
                    folders,
                    is_positive
                )
            
                if not success:
                    print("❌ Failed to process all Augustus chunks")
                else:
                    print("✅ All Augustus chunks processed successfully!")

            # Close GenBank tab and return to main handle
            print("🔄 Returning to main tab...")
            driver.close()
            driver.switch_to.window(main_handle)
            time.sleep(2)

        return None, None

    except Exception as e:
        print(f"❌ Error processing CSV: {e}")
        import traceback
        traceback.print_exc()
        return None, None


# ---------- Pipeline ----------
def run_pipeline(species_name, fasta_seq):
    # Create folder structure first
    folders = get_species_folder(species_name)
    
    edge_opts = get_edge_options()
    driver = webdriver.Edge(service=Service(EDGEDRIVER_PATH), options=edge_opts)

    try:
        print("🌐 Opening genome browser…")
        driver.get(GENOME_HOME)
        wait_ready(driver)

        # ✅ Search species & open results
        results_url = get_results_url(driver, species_name)
        driver.get(results_url)
        wait_ready(driver)

        # ✅ Choose assembly by priority
        load_all_rows(driver)
        before = driver.current_url
        clicked = pick_and_click_by_priority(driver)

        if clicked:
            if not wait_url_change(driver, before, timeout=TIMEOUT):
                time.sleep(2)
            wait_ready(driver)

            # ✅ BLAST workflow
            click_blast_button(driver)
            wait_ready(driver)
            set_tblastn(driver)
            paste_query_sequence(driver, fasta_seq)
            click_final_blast(driver)
            click_alignments_button(driver)

            # ✅ Download + process CSV
            click_download_button(driver)
            time.sleep(3)
            before_files = set(glob.glob(os.path.join(BASE_DIR, "*.csv")))
            download_hit_table(driver)
            csv_path = rename_latest_csv(species_name, before_files, folders)

            # ✅ Process CSV and download full FASTA → AUGUSTUS
            if csv_path:
                begin, end = process_csv(species_name, driver, folders)
                if begin and end:
                    print(f"✅ CSV processed with BEGIN={begin} and END={end}")
                else:
                    print("⚠️ CSV processed but BEGIN/END not extracted (mixed case or error)")

        else:
            print("🚫 Nothing clicked (no priority matched).")

        print("📌 Final URL:", driver.current_url)
        input("\nPress Enter to close browser…")
    except Exception as e:
        print(f"\n❌ ERROR: {e}")
        traceback.print_exc()
        input("\nPress Enter after checking error…")
    finally:
        driver.quit()


if __name__ == "__main__":
    species = input("Enter species name: ").strip()
    fasta = """>sp|O04714|GCR1_ARATH G-protein coupled receptor 1 OS=Arabidopsis thaliana OX=3702 GN=GCR1 PE=1 SV=1
MSAVLTAGGGLTAGDRSIITAINTGASSLSFVGSAFIVLCYCLFKELRKFSFKLVFYLAL
SDMLCSFFLIVGDPSKGFICYAQGYTTHFFCVASFLWTTTIAFTLHRTVVKHKTDVEDLE
AMFHLYVWGTSLVVTVIRSFGNNHSHLGPWCWTQTGLKGKAVHFLTFYAPLWGAILYNGF
TYFQVIRMLRNARRMAVGMSDRVDQFDNRAELKVLNRWGYYPLILIGSWAFGTINRIHDF
IEPGHKIFWLSVLDVGTAALMGLFNSIAYGFNSSVRRAIHERLELFLPERLYRWLPSNFR
PKNHLILHQQQQQRSEMVSLKTEDQQ"""
    run_pipeline(species, fasta)

Enter species name:  malus domestica



📁 Folder structure created for malus_domestica:
   📊 Excel files: D:\DOWNLOAD_SECTION\Augustus_csv\malus_domestica\excel
   🧬 FASTA files: D:\DOWNLOAD_SECTION\Augustus_csv\malus_domestica\fasta
   🧪 Protein results: D:\DOWNLOAD_SECTION\Augustus_csv\malus_domestica\protein
   🧪 Nucleotide results: D:\DOWNLOAD_SECTION\Augustus_csv\malus_domestica\nucleotide
🌐 Opening genome browser…
🔎 Selecting species…
⚠️ Exact match not found — first suggestion used
🔗 Results URL: https://www.ncbi.nlm.nih.gov/datasets/genome/?taxon=3750
📜 Loading all rows…
✅ Scrolling done
🧮 Rows detected: 20 (table)
👉 Priority 2 (Badge only): Clicking assembly  →  GDT2T_hap1  (https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_042453785.1/)
🚀 Looking for BLAST button…
✅ BLAST button clicked.
🎯 Setting tBLASTn…
✅ tBLASTn set (methods tried: direct id tblastnTab).
📝 Pasting query sequence…
✅ Query pasted successfully.
🔥 Clicking final BLAST button…
✅ Final BLAST started.
📊 Waiting for BLAST results and Alignments button…


Press Enter to close browser… 
